# Laboratory Work 3: Building a Custom Image Classifier with TensorFlow

This notebook will guide you through the process of building, training, and evaluating a custom image classifier using your own image dataset from Google Drive.

## Part 1: Preparing and Loading Custom Images from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Define Dataset Path and Load Images
Set the path to your image dataset and use TensorFlow to load the images. Make sure your dataset is organized into subdirectories, where each subdirectory represents a class.

In [ ]:
import tensorflow as tf

dataset_path = "/content/drive/MyDrive/Plants Img/images"

img_height = 180
img_width = 180
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

Found 5001 files belonging to 20 classes.
Using 4001 files for training.
Found 5001 files belonging to 20 classes.
Using 1000 files for validation.


### View Class Names
Check the class names to ensure the dataset was loaded correctly.

In [ ]:
class_names = train_ds.class_names
print(class_names)

['01_princess_flower', '02_angels_trumpet', '03_bleeding_heart_vine', '04_firecracker_plant', '05_bluesky_vine', '06_mandevilla', '07_golden_dewdrop', '08_mexican_petunia', '09_firebush', '10_coreopsis', '11_calla_lily', '12_blue_passion_flower', '13_morning_glory', '14_flame_lily', '15_bird_of_paradise', '16_madagascar_periwinkle', '17_garden_cosmos', '18_cape_plumbago', '19_scarlet_sage', '20_spider_flower']


## Part 2: Training and Evaluating the Image Classification Model

### Optimize Dataset Performance
To improve performance, use buffered prefetching to load data from disk without having it be a bottleneck.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Build the CNN Model
Create a simple Convolutional Neural Network (CNN) for image classification.

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names))
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Compile the Model
Compile the model with an optimizer, loss function, and metrics.

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

### Train the Model
Train the model for a fixed number of epochs.

In [ ]:
epochs = 10
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

Epoch 1/10


### Evaluate Model Performance
Check the model's performance on the validation set.

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print("Validation Accuracy:", accuracy)

### Test with a New Image
Upload a new image to your Google Drive and test the model's prediction. Make sure to change `img_path` to the correct path of your test image.

In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
import os

test_images_path = "/content/drive/MyDrive/test_images"
test_image_files = [f for f in os.listdir(test_images_path) if os.path.isfile(os.path.join(test_images_path, f))]

for image_file in test_image_files:
    img_path = os.path.join(test_images_path, image_file)
    img = load_img(img_path, target_size=(img_height, img_width))
    img_array = img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    print(f"Image: {image_file}")
    print("Predicted Class:", class_names[np.argmax(score)])
    print("-" * 30)

# Activity 3A: Improving and Evaluating a Custom Image Classifier

## Part 3: Visualizing Training Results & Detecting Overfitting

### Plot Training vs Validation Accuracy and Loss
Visualize the model's performance to check for overfitting. Overfitting occurs when the training accuracy is much higher than the validation accuracy.

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## Part 4: Applying Data Augmentation

### Create Data Augmentation Layer
Data augmentation generates more training data from your existing images by applying random transformations. This helps expose the model to more aspects of the data and prevent overfitting.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

### Visualize Augmented Images
Let's visualize what a few augmented examples look like by applying data augmentation to the same image several times.

In [ ]:
import matplotlib.pyplot as plt

for images, _ in train_ds.take(1):
    plt.figure(figsize=(8, 8))
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype("uint8"))
        plt.axis("off")
    plt.show()

## Part 5: Reducing Overfitting Using Dropout

### Build an improved CNN Model
Create a new model that includes the data augmentation layer and dropout layers to reduce overfitting.

In [ ]:
model = models.Sequential([
    data_augmentation,
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names))
])

## Part 6: Compile and Train the Improved Model

### Compile Model
Compile the new model.

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

### Train Model
Train the improved model. You may need to train for more epochs to see the benefit of data augmentation and dropout.

In [ ]:
epochs = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

### Visualize Improved Training Results
Plot the training and validation accuracy and loss again to see if overfitting has been reduced.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## Part 7: Predict on New Data

### Predict on New Data
Use the improved model to make a prediction on a new image. Remember to change the `img_path`.

In [ ]:
import os
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array

test_images_path = "/content/drive/MyDrive/test_images"
test_image_files = [f for f in os.listdir(test_images_path) if os.path.isfile(os.path.join(test_images_path, f))]

for image_file in test_image_files:
    img_path = os.path.join(test_images_path, image_file)
    img = load_img(img_path, target_size=(img_height, img_width))
    img_array = img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    print(f"Image: {image_file}")
    print("Predicted Class:", class_names[np.argmax(score)])
    print("Confidence:", round(100 * np.max(score), 2), "%")
    print("-" * 30)

## Part 8: Save and Reuse the Model

### Save Model to Google Drive
Save your trained model to Google Drive so you can reuse it later without having to retrain.

In [ ]:
model.save("/content/drive/MyDrive/my_image_classifier.keras")

### Load Saved Model
Load the model from Google Drive.

In [ ]:
from tensorflow.keras.models import load_model

loaded_model = load_model("/content/drive/MyDrive/my_image_classifier.keras")

In [ ]:
import tensorflow as tf
import builtins

# Inject tf as a global so Lambda can find it during deserialization
builtins.tf = tf

tf.keras.config.enable_unsafe_deserialization()

m = tf.keras.models.load_model("/content/drive/MyDrive/my_image_classifier_good.keras")

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/drive/MyDrive/Plants Img/images",
    validation_split=0.2, subset="validation",
    seed=123, image_size=(180, 180), batch_size=32)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

m.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
loss, acc = m.evaluate(val_ds, verbose=1)
print(f"LW4 Good Model → Test Loss: {loss:.4f} | Test Acc: {acc*100:.2f}%")


Found 5001 files belonging to 20 classes.
Using 1000 files for validation.
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 194ms/step - accuracy: 0.8000 - loss: 0.8264
LW4 Good Model → Test Loss: 0.8264 | Test Acc: 80.00%
